# Detect Unity Catalog schema changes

Offline path: use the included metadata-only fixture. Databricks path: use the active `spark` session. No credentials or source records are embedded.

## Prerequisites

Install `great-generator`. For SDK access outside a notebook, install `great-generator[databricks]` and use Databricks unified authentication. Metadata visibility and source object permissions are controlled by Unity Catalog.

In [ ]:
from pathlib import Path

from great_generator import StaticUnityCatalogClient

SOURCE_TABLE = "retail.sales.fact_orders"
FIXTURE = Path("examples/databricks/unity_catalog/mock_uc_metadata.json")
client = StaticUnityCatalogClient.from_path(FIXTURE)

# In a Databricks workspace, remove client=client and pass spark=spark instead.


## Generate synthetic data from metadata

In [ ]:
from copy import deepcopy

from great_generator import UnityCatalogSnapshot, compare_uc_schema_snapshots

old = client.snapshot
payload = deepcopy(old.as_dict())
payload["tables"][SOURCE_TABLE]["columns"].append({
    "name": "channel",
    "position": 8,
    "data_type": "STRING",
    "nullable": True,
})
new = UnityCatalogSnapshot.from_dict(payload)
diff = compare_uc_schema_snapshots(old, new)
print(diff.to_text())


## Validate

In [ ]:
assert diff.changed is True
assert diff.added_columns[SOURCE_TABLE] == ("channel",)
# Snapshot comparison reports drift; it never migrates source schemas.


## Cleanup and limitations

These examples do not write tables. Remove only sandbox artifacts you explicitly create. Generated data is synthetic, not anonymized production data. Complex types and cyclic/self-referencing relationships require explicit modeling.